# Resumen UT3 – APIs, JSON, XML y Requests

Objetivo: recopilar ejemplos mínimos que me permitan en el examen:
- Hacer peticiones HTTP (GET/POST) con `requests`
- Trabajar con JSON (serializar / deserializar)
- Consumir APIs públicas (REST Countries, APIs de ejemplo)
- Parsear XML básico con `xml.etree.ElementTree`
- Devolver resultados limpios (listas, diccionarios, clases)


## Cosas

- `requests.get(url, params=...)` → peticiones GET
- `response.status_code`, `response.ok`, `response.text`
- `response.json()` → deserializa JSON a dict/list

- `json.dump(obj, f)` → guardar a archivo
- `json.load(f)` → leer desde archivo
- `json.dumps(obj)` / `json.loads(cadena)` → trabajar con strings

- Filtrado:
  - List comprehension: `[p for p in paises if p['region'] == 'Europe']`
  - `.get('clave', valor_defecto)` para evitar `KeyError`

- XML:
  - `ET.fromstring(xml_string)`
  - `root.find('.//tag')`, `.get('atributo')`

- Errores típicos:
  - Escribir mal la URL
  - Olvidar `()` en `.json()`
  - Suponer que siempre existe `capital`, `currencies`, etc. → usar `get`


In [ ]:
import requests
import json
import xml.etree.ElementTree as ET
from dataclasses import dataclass

def pretty(obj):
    """Imprime JSON/dict de forma legible."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))


In [ ]:
# 3. Petición GET simple + control de errores

# GET simple a una API pública
url = "https://restcountries.com/v3.1/all"

response = requests.get(url)

print("Código de estado:", response.status_code)
if response.ok:
    data = response.json()  # deserializar JSON -> objetos Python (lista/dicts)
    print("Número de países:", len(data))
    print("Primer país (formato crudo):")
    pretty(data[0])
else:
    print("Error en la petición:", response.text)


In [ ]:
# 4 PARAMETROS DE CONSULTA

# Ejemplo: búsqueda por nombre con parámetros
url_name = "https://restcountries.com/v3.1/name/spain"

params = {
    "fullText": "true"  # ?fullText=true
}

response = requests.get(url_name, params=params)
pais = response.json()[0]

print("Nombre común:", pais["name"]["common"])
print("Capital:", pais.get("capital", ["N/A"])[0])
print("Región:", pais["region"])


In [ ]:
# 5. FILTRADO API

# Pedir solo algunos campos (más eficiente)
url_fields = "https://restcountries.com/v3.1/region/europe"
params = {"fields": "name,capital,population,region"}

response = requests.get(url_fields, params=params)
paises_europa = response.json()

print("Total países Europa:", len(paises_europa))
print("Ejemplo de un país (ya filtrado por la API):")
pretty(paises_europa[0])

# Filtrado adicional en Python: países con > 50M habitantes
grandes = [p for p in paises_europa if p.get("population", 0) > 50_000_000]
print("Países europeos con >50M habitantes:")
for p in grandes:
    print("-", p["name"]["common"], "→", p["population"])


In [ ]:
# 6. SERIALIZACIÓN Y DESERIALIZACIÓN

# Serializar: guardar lista/dict en un archivo JSON
with open("paises_europa.json", "w", encoding="utf-8") as f:
    json.dump(paises_europa, f, indent=2, ensure_ascii=False)

# Deserializar: leer de archivo JSON
with open("paises_europa.json", "r", encoding="utf-8") as f:
    paises_europa_cargados = json.load(f)

print("Cargados desde archivo:", len(paises_europa_cargados))


In [ ]:
#7 PROCESAR RESPUESTAS

# Crear un resumen sencillo: nombre, capital, población
resumen = []
for p in paises_europa_cargados:
    nombre = p["name"]["common"]
    capital = p.get("capital", ["N/A"])[0] if p.get("capital") else "N/A"
    poblacion = p.get("population", 0)
    resumen.append({
        "nombre": nombre,
        "capital": capital,
        "habitantes": poblacion
    })

print("Primeros 5 países resumidos:")
pretty(resumen[:5])


In [ ]:
# Deserializar objetos

@dataclass
class Pais:
    nombre: str
    capital: str
    poblacion: int
    region: str

    def info(self):
        print(f"{self.nombre} ({self.region}) – {self.capital}, {self.poblacion:,} hab.")

# Deserializar lista de dicts -> lista de objetos Pais
paises_obj = []
for p in paises_europa_cargados:
    nombre = p["name"]["common"]
    capital = p.get("capital", ["N/A"])[0] if p.get("capital") else "N/A"
    poblacion = p.get("population", 0)
    region = p.get("region", "N/A")
    paises_obj.append(Pais(nombre, capital, poblacion, region))

print("Total objetos Pais:", len(paises_obj))
for pais in paises_obj[:5]:
    pais.info()


In [ ]:
# 9  POST CON JSON

url_post = "https://httpbin.org/post"  # API de test

payload = {
    "nombre": "Alvaro",
    "edad": 22,
    "skills": ["python", "apis", "json"]
}
response = requests.post(url_post, json=payload)
data = response.json()

print("Datos que ha recibido el servidor (echo):")
pretty(data["json"])


In [ ]:
# 10 XML

# Hardcodeado. Podria ser la respuesta de una API que devuelve XML, o un archivo XML local.
xml_string = """ 
<root>
    <city name="Elche">
        <temperature value="22.5" unit="C"/>
        <humidity value="60" unit="%"/>
    </city>
</root>
"""

# Deserializar XML desde cadena
root = ET.fromstring(xml_string)

city = root.find(".//city")
name = city.get("name")
temperature = city.find("temperature").get("value")
humidity = city.find("humidity").get("value")

print(f"Ciudad: {name}")
print(f"Temperatura: {temperature} °C")
print(f"Humedad: {humidity} %")


In [ ]:
# 11. PLANTILLA EJERCICIO API
def ejercicio_api(
    url: str,
    params: dict | None = None,
    descripcion: str = "Ejercicio API"
):
    print(f"=== {descripcion} ===")
    print("URL:", url)
    if params:
        print("Params:", params)
    resp = requests.get(url, params=params)
    print("Status:", resp.status_code)
    if not resp.ok:
        print("Error:", resp.text[:200])
        return None
    data = resp.json()
    print("Tipo de dato:", type(data))
    if isinstance(data, list):
        print("Elementos:", len(data))
        if data:
            print("Primer elemento (vista previa):")
            pretty(data[0])
    elif isinstance(data, dict):
        print("Claves:", list(data.keys())[:10])
    return data

# Ejemplo de uso:
_ = ejercicio_api("https://restcountries.com/v3.1/lang/spa",
                descripcion="Países que hablan español")



In [ ]:
# ============================================
# CELDA: JSON – Crear archivo + Leer archivo
# ============================================

import json
import requests

print("🔄 EJEMPLO COMPLETO: APIs → JSON → Archivo → Leer")

# ============================================
# PASO 1: Obtener datos de API
# ============================================
print("\n1. PETICIÓN A API...")
url = "https://restcountries.com/v3.1/region/europe?fields=name,capital,population"
response = requests.get(url)

if response.ok:
    paises = response.json()
    print(f"✅ Datos obtenidos: {len(paises)} países")
else:
    print("❌ Error en API")
    paises = []

# ============================================
# PASO 2: CREAR ARCHIVO JSON (SERIALIZAR)
# ============================================
print("\n2. CREANDO ARCHIVO paises_europa.json...")

# json.dump(objeto_python, archivo, indent=2, ensure_ascii=False)
with open("paises_europa.json", "w", encoding="utf-8") as archivo:
    json.dump(
        paises,           # objeto Python (lista de dicts)
        archivo,          # archivo de destino
        indent=2,         # formato legible
        ensure_ascii=False  # caracteres especiales (ñ, á, etc.)
    )

print("✅ Archivo 'paises_europa.json' CREADO")

# ============================================
# PASO 3: LEER ARCHIVO JSON (DESERIALIZAR)
# ============================================
print("\n3. LEYENDO ARCHIVO paises_europa.json...")

# json.load(archivo) → objeto Python
with open("paises_europa.json", "r", encoding="utf-8") as archivo:
    paises_cargados = json.load(archivo)  # ¡Ya es lista de dicts!

print(f"✅ Archivo LEÍDO: {len(paises_cargados)} países")
print("Primer país cargado:")
print(json.dumps(paises_cargados[0], indent=2, ensure_ascii=False)[:300] + "...")

# ============================================
# PASO 4: TRABAJAR CON LOS DATOS CARGADOS
# ============================================
print("\n4. TRABAJANDO con datos cargados...")

# Ejemplo: filtrar países con > 50M habitantes
grandes = [
    p["name"]["common"] 
    for p in paises_cargados 
    if p.get("population", 0) > 50_000_000
]

print("Países con >50M habitantes:")
for nombre in grandes:
    print(f"• {nombre}")
